In [ ]:
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q evaluate sacrebleu rouge-score bert-score sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [ ]:
import unsloth
import torch

import transformers
import datasets


print("All imports successful!")
print(torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
All imports successful!
Tesla T4


In [ ]:
from unsloth import FastLanguageModel

print("Unsloth ready!")

Unsloth ready!


In [ ]:
from datasets import load_dataset
dataset=load_dataset("cfilt/iitb-english-hindi")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [ ]:
print(dataset["train"][0])

{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [ ]:
dataset=dataset["train"].shuffle(seed=42)

In [ ]:
train_dataset=dataset.select(range(5000))
val_dataset=dataset.select(range(5000,5500))

In [ ]:
def filter_empty(example):
    en = example["translation"]["en"]
    hi = example["translation"]["hi"]

    return (
        en is not None and hi is not None and
        len(en.strip()) > 0 and
        len(hi.strip()) > 0
    )

train_dataset = train_dataset.filter(filter_empty)
val_dataset = val_dataset.filter(filter_empty)

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
def format_prompt(example):
    en = example["translation"]["en"]
    hi = example["translation"]["hi"]

    text = f"""Translate English to Hindi:

English: {en}

Hindi: {hi}{tokenizer.eos_token}"""

    return {"text": text}

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = 256,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
train_dataset = train_dataset.map(format_prompt)
val_dataset = val_dataset.map(format_prompt)

Map:   0%|          | 0/4978 [00:00<?, ? examples/s]

Map:   0%|          | 0/497 [00:00<?, ? examples/s]

In [ ]:
def filter_long_examples(example):
    return len(tokenizer(example["text"])["input_ids"]) <= 256

train_dataset = train_dataset.filter(filter_long_examples)
val_dataset = val_dataset.filter(filter_long_examples)

print(len(train_dataset))
print(len(val_dataset))
print(train_dataset[0]["text"])

Filter:   0%|          | 0/4978 [00:00<?, ? examples/s]

Filter:   0%|          | 0/497 [00:00<?, ? examples/s]

4712
468
Translate English to Hindi:

English: on the intuition.

Hindi: अंतर्ज्ञान पर। <|im_end|>


In [ ]:
tokenizer.pad_token = tokenizer.eos_token
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
FastLanguageModel.for_training(model)
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir = "outputs",


    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,


    num_train_epochs = 1,


    learning_rate = 2e-4,


    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),


    logging_steps = 10,

    optim = "adamw_8bit",


    weight_decay = 0.01,


    lr_scheduler_type = "linear",


    seed = 42,


    save_strategy = "epoch",


    report_to = "none",
)

In [ ]:


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,

    train_dataset = train_dataset,
    eval_dataset = val_dataset,

    dataset_text_field = "text",

    max_seq_length = 256,

    args = training_args,
)

print("Trainer is ready!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4712 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/468 [00:00<?, ? examples/s]

Trainer is ready!


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,712 | Num Epochs = 1 | Total steps = 589
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.816114
20,1.653144
30,1.611522
40,1.646429
50,1.555369
60,1.506361
70,1.571541
80,1.494250
90,1.572529
100,1.455794


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-589/tokenizer_config.json.


In [ ]:
model.save_pretrained("qwen_lora_model")
tokenizer.save_pretrained("qwen_lora_model")

print("Model saved successfully!")
FastLanguageModel.for_inference(model)

Unsloth: Restored added_tokens_decoder metadata in qwen_lora_model/tokenizer_config.json.


Model saved successfully!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [ ]:
prompt = """Translate English to Hindi:

English: How are you?

Hindi:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 64,
    temperature = 0.2,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Translate English to Hindi:

English: How are you?

Hindi: क्या आप अच्छे है? 


In [ ]:
predictions = []
references = []
english_sentences = []

for example in val_dataset:

    en = example["translation"]["en"]
    hi = example["translation"]["hi"]

    prompt = f"""Translate English to Hindi:

English: {en}

Hindi:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        do_sample = False,
        temperature = 0.0,
        eos_token_id = tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    prediction = decoded.split("Hindi:")[-1].strip()

    english_sentences.append(en)
    references.append(hi)
    predictions.append(prediction)

print("Prediction generation complete!")

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Prediction generation complete!


In [33]:
from evaluate import load


bleu = load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)


rouge = load("rouge")
rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)


bertscore = load("bertscore")
bert_score = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="hi"
)

print("BLEU Score:", bleu_score["bleu"])

print("\nROUGE Scores:")
print(rouge_score)

print("\nBERTScore F1 Average:")
print(sum(bert_score["f1"]) / len(bert_score["f1"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLEU Score: 0.04114713735982988

ROUGE Scores:
{'rouge1': np.float64(0.059971509971509976), 'rouge2': np.float64(0.017094017094017096), 'rougeL': np.float64(0.0584045584045584), 'rougeLsum': np.float64(0.05947293447293448)}

BERTScore F1 Average:
0.7522991000332384


In [34]:
import pandas as pd

results_df = pd.DataFrame({
    "English": english_sentences,
    "Reference_Hindi": references,
    "Predicted_Hindi": predictions
})

results_df.to_csv("translation_results.csv", index=False,encoding="utf-8-sig")

print(results_df.head())

print("\nCSV saved successfully!")

                                             English  \
0                                       Display unit   
1                                        State Value   
2  These are nutritional factors, such as plant f...   
3  India has successfully met the freeze as on 1....   
4  The programme has also focused on awareness of...   

                                     Reference_Hindi  \
0                                        प्रदर्श एकक   
1                                    कौशल का खेलName   
2  ये हैं पोषण संबंधी कारक जैसे पौधों से प्राप्त ...   
3  मॉन्ट्रियल प्रोटोकॉल के त्वरित चरण आउट शेड्यूल...   
4  कार्यक्रम में बच्चों खासकर लड़कियों की शिक्षा ...   

                                     Predicted_Hindi  
0                                इकाई प्रदर्शित करें  
1                                        राज्य मूल्य  
2  ये प्राकृतिक संकेतों की बार्षिक आवश्यकता है, ज...  
3  भारत को मैलिनी परमाणु संधार के मॉक्सनप्रोटी का...  
4  यह प्रोग्राम का भी अंतर्गत बच्चों को समान मूल्..